In [1]:
import requests
from transformers import pipeline
import torch

In [2]:
def load_gpt_pipeline(model_name="EleutherAI/gpt-neo-2.7B"):
    device = 0 if torch.cuda.is_available() else -1
    print(f"Using {'GPU' if device == 0 else 'CPU'} for inference.")
    return pipeline("text-generation", model=model_name, device=device)

In [3]:
model_name = "EleutherAI/gpt-neo-2.7B"
text_gen_pipeline = load_gpt_pipeline(model_name)

Using GPU for inference.


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.7G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Device set to use cuda:0


In [4]:
def get_wikipedia_data(species_name):
    print("Fetching information from Wikipedia...")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{species_name}"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            return response.json().get("extract", "No information available.")
        return "No data found on Wikipedia."
    except requests.exceptions.RequestException as error:
        return f"Error fetching data: {error}"

In [ ]:
def create_educational_content(pipeline, species_name, facts, max_length=128):
    facts = facts[:500]
    prompt = (f"Write an educational article about the species {species_name}. "
              f"Include its habitat, threats, conservation status, and interesting facts. "
              f"Details: {facts}")
    try:
        print("Generating educational content...")
        result = pipeline(
            prompt,
            max_new_tokens=max_length,
            truncation=True,
            num_return_sequences=1,
            pad_token_id=pipeline.tokenizer.eos_token_id
        )
        return result[0]["generated_text"]
    except Exception as error:
        return f"Error generating content: {error}"

In [6]:
print("---------------Welcome to the Wildlife Education Generator-----------------")
species_name = input("Enter the name of the species you'd like to learn about: ").strip()
wiki_data = get_wikipedia_data(species_name)
print("\nWikipedia Summary:")
print(wiki_data)
educational_content = create_educational_content(text_gen_pipeline, species_name, wiki_data)
print("\nGenerated Educational Content:")
print(educational_content)

---------------Welcome to the Wildlife Education Generator-----------------
Enter the name of the species you'd like to learn about: cheetah
Fetching information from Wikipedia...

Wikipedia Summary:
The cheetah is a large cat and the fastest land animal. It has a tawny to creamy white or pale buff fur that is marked with evenly spaced, solid black spots. The head is small and rounded, with a short snout and black tear-like facial streaks. It reaches 67–94 cm (26–37 in) at the shoulder, and the head-and-body length is between 1.1 and 1.5 m. Adults weigh between 21 and 65 kg. The cheetah is capable of running at 93 to 104 km/h ; it has evolved specialized adaptations for speed, including a light build, long thin legs and a long tail.
Generating educational content...

Generated Educational Content:
Write an educational article about the species cheetah. Include its habitat, threats, conservation status, and interesting facts. Details: The cheetah is a large cat and the fastest land anim